In [1]:
import depthai as dai
import numpy as np
import json



In [2]:
def build_homogeneous(rotation_matrix, translation_vector):
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)
    return T_camera_to_base_effector

def convert_coordinates(x ,y ,z, homogeneous_matrix): # X Y Z coordinates that should be translated into robot frame coordinates
    obj_camera_coordinates = np.array([x, y, z])
    obj_camera_coordinates_homo = np.append(obj_camera_coordinates, [1])  # Convert object coordinates to homogeneous coordinates
    obj_base_effector_coordinates_homo = homogeneous_matrix.dot(obj_camera_coordinates_homo)
    obj_base_coordinates = obj_base_effector_coordinates_homo[:3]  
    
    #return list(map(int, obj_base_coordinates)) # Uncomment this line if you want to send integers instead of floats
    return np.around(obj_base_coordinates,2).tolist() # Use this to get a list of new coordinates, chane the number to get the number of decimal numbers

def estimate_rigid_transform(camera_points, robot_points):
    cam = np.asarray(camera_points, dtype=np.float64)
    rob = np.asarray(robot_points,  dtype=np.float64)
    assert cam.shape == rob.shape and cam.shape[1] == 3 and cam.shape[0] >= 3, "Value error, differing amount of coordinates, Camera:" + str(len(cam)) + " Robot:"+ str(len(rob))
    

    camera_centroid = cam.mean(axis=0)
    robot_centroid  = rob.mean(axis=0)
    camera_centered = cam - camera_centroid
    robot_centered  = rob - robot_centroid

    cross_covariance = camera_centered.T @ robot_centered
    U, s, Vt = np.linalg.svd(cross_covariance)
    V = Vt.T

    # Ensure of proper rotation (det=+1)
    det_correction = np.sign(np.linalg.det(V @ U.T))
    rotation_matrix = V @ np.diag([1.0, 1.0, det_correction]) @ U.T

    translation_vector = robot_centroid - rotation_matrix @ camera_centroid
    return rotation_matrix, translation_vector

def extract_data(file):
    
    float_list=[]
    with open(file, "r") as f:
        lines = f.readlines()
        for i in lines:
            x = json.loads(i)
            float_list.append(x)
    return list(float_list)

In [ ]:
cam_coords = extract_data("saved_coordinates.txt")
robot_coords = extract_data("robo_coords.txt")

print(cam_coords)
print(robot_coords)



[[-94.75338745117188, -26.71119499206543, 770.5167236328125], [-233.3564453125, 155.6881103515625, 1320.8858642578125], [-50.252891540527344, 174.82017517089844, 1381.6162109375], [-20.19759178161621, 23.32246208190918, 897.0194702148438], [107.1271743774414, 165.86874389648438, 1166.9962158203125], [197.31492614746094, 77.74153900146484, 897.0194702148438], [-59.799407958984375, 164.01809692382812, 1278.7298583984375], [-47.316036224365234, 31.56777572631836, 910.6106567382812], [27.635913848876953, 75.59503936767578, 1063.72216796875], [-281.6695556640625, 106.2162857055664, 1178.4373779296875], [16.170141220092773, -8.091158866882324, 777.9974975585938], [-253.22769165039062, 37.41748046875, 981.2294921875], [-228.3380126953125, 145.6190948486328, 1292.4796142578125], [24.784589767456055, 39.685203552246094, 953.9730834960938], [-70.53450012207031, 157.11436462402344, 1313.6678466796875], [-209.91111755371094, 41.32509231567383, 993.3934936523438], [66.24244689941406, 33.14616775512

In [7]:
R, t = estimate_rigid_transform(cam_coords, robot_coords)
print("R =\n", R)
print("t =", t)
homogeneous = build_homogeneous(R,t)
print(homogeneous)
print(convert_coordinates(100,100,100, homogeneous))

R =
 [[-0.91834122 -0.27045221 -0.28897233]
 [ 0.35897406 -0.26165042 -0.89592225]
 [ 0.16669442 -0.9264959   0.3373697 ]]
t = [ 640.5575017  1070.72414756 -225.70844268]
[[-9.18341217e-01 -2.70452213e-01 -2.88972332e-01  6.40557502e+02]
 [ 3.58974063e-01 -2.61650421e-01 -8.95922251e-01  1.07072415e+03]
 [ 1.66694423e-01 -9.26495903e-01  3.37369696e-01 -2.25708443e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
[492.78, 990.86, -267.95]


In [24]:
# index = 0
# try:
#     with open('RT.txt', 'r') as test:
#         for j in test:
#             index += 1
#             print(index)   
# except:
#     pass

with open('RT.txt', 'a') as f:
    for i in [R,t]:
        f.write("%s\n" % i) #saves the camera coordinates to a .txt file